# Single-game odds time series visualizer

Adapted from the reference notebook into reusable modules in `src/analysis/kalshi/`.
Run the cells below, click **Find local games**, filter by sport/team/ticker, choose
an outcome, and click **Visualize game**. You can change games without rerunning cells.

Alternatively, enter an event or child market ticker in **Ticker override** and
select **Kalshi API**. No sportsbook or play-by-play files are required.

Use the repo's Python environment with `requirements.txt` installed. Notebook
controls additionally need `pip install ipywidgets ipykernel` (and Jupyter or VS Code).


In [ ]:
from pathlib import Path
from IPython.display import display
from src.analysis.kalshi.notebook_game_picker import game_picker

# Directory containing kalshi/markets and kalshi/trades_global_staging.
DATA_DIR = Path("data").resolve()
display(game_picker(DATA_DIR))


## Data and time behavior

- The catalog lists game-winner markets. **Ticker override** also accepts events
  absent from the local catalog; choose the API source to fetch their trades.
- Local charts use stored trade prices in cents and repair them using the same
  authoritative taker-side price rule as the backtester. They include pregame
  trading unless you enter a start time. Close time is not treated as kickoff.
- API charts use Kalshi's scheduled game start. If unavailable, enter a start.
- Times are UTC; timestamps without an offset are interpreted as UTC.
- Two-outcome events combine both markets into the selected team's probability.
  Events with one or three-plus outcomes use only the selected market's YES trades.
- Fee lines preserve the reference notebook's estimates; they are not a fee ledger.
- The first catalog scan and each local trade scan can take time on a large archive.
  Use a larger sample interval or a narrower time window for long histories.
- Each chart is also saved as standalone interactive HTML in `output/`.

## Add overlays later

Drop an event-specific file at `data/playbyplay/<EVENT_TICKER>.csv` or
`data/sportsbook/<EVENT_TICKER>.csv` and visualize again. You can also supply a
CSV/Parquet path in the controls. Absent or empty optional files produce a normal
Kalshi chart; an explicitly entered missing path reports an error.

Canonical play-by-play needs `timestamp`; optional columns are `event_type`,
`event_label`, `event_detail`, `score_text`, `game_clock_text`, `down_distance_text`.
Types `Scoring play`, `Quarter end`, `Period end`, and `2-minute warning` enable
the corresponding markers and period lines. Other types appear on the plays track.
The original SportsDataIO `Created`/`Description`/`IsScoringPlay` format also works.

Sportsbook files use `timestamp,book,target_team,implied_prob_raw` (probabilities
from 0 to 1), or `raw_odds` in American format instead of `implied_prob_raw`.
For the reference SportsDataIO `Sportsbook,Updated,HomeMoneyLine,AwayMoneyLine`
format, select whether your chosen outcome is **Home** or **Away**.
Use **Sportsbook team** when a provider's team name differs from Kalshi's.
See `examples/game_overlays/` for schema-only templates.


## Use the same chart from Python

The controls call the same function as scripts. For example:

```python
from src.analysis.kalshi.single_game_odds_time_series import visualize_game_odds

figure = visualize_game_odds(
    "KXNFLGAME-25SEP07CINCLE", target="CIN", source="local",
    start_time="2025-09-07T17:00:00Z", resample_frequency="10s", show=False,
)
figure.show()
```
